In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import importlib
import utils
importlib.reload(utils)
from utils import *

import torch
import warnings
warnings.filterwarnings("ignore")


In [3]:
# ── Step 1: verify imports ────────────────────────────────────────────────────
print("\n--- Step 1: Import Check ---")
classes = [
    # building blocks
    "autopad", "Conv", "Bottleneck", "C2f", "C3", "C3k", "C3k2", "SPPF",
    # neck
    "Attention", "PSABlock", "C2PSA", "Concat",
    # head
    "DWConv", "DFL", "make_anchors", "dist2bbox", "Detect",
    # model
    "model_scales", "get_model_params", "Backbone", "Neck", "DetectHead", "YOLOv11"
]
all_good = True
for cls in classes:
    exists = cls in dir()
    status = "✅" if exists else "❌"
    print(f"  {status} {cls}")
    if not exists:
        all_good = False


--- Step 1: Import Check ---
  ✅ autopad
  ✅ Conv
  ✅ Bottleneck
  ✅ C2f
  ✅ C3
  ✅ C3k
  ✅ C3k2
  ✅ SPPF
  ✅ Attention
  ✅ PSABlock
  ✅ C2PSA
  ✅ Concat
  ✅ DWConv
  ✅ DFL
  ✅ make_anchors
  ✅ dist2bbox
  ✅ Detect
  ✅ model_scales
  ✅ get_model_params
  ✅ Backbone
  ✅ Neck
  ✅ DetectHead
  ✅ YOLOv11


In [4]:
# ── Step 2: verify model_scales ───────────────────────────────────────────────
print("\n--- Step 2: model_scales Check ---")
for name, (d, w, mc) in model_scales.items():
    print(f"  ✅ {name:<10} d={d}, w={w}, mc={mc}")


--- Step 2: model_scales Check ---
  ✅ yolo11n    d=0.5, w=0.25, mc=1024
  ✅ yolo11s    d=0.5, w=0.5, mc=1024
  ✅ yolo11m    d=0.5, w=1.0, mc=512
  ✅ yolo11l    d=1.0, w=1.0, mc=512
  ✅ yolo11x    d=1.0, w=1.5, mc=512


In [5]:
# ── Step 3: verify get_model_params ──────────────────────────────────────────
print("\n--- Step 3: get_model_params Check ---")
ch, dep = get_model_params("yolo11n")
checks = [
    (ch(64),    16,  "ch(64)"),
    (ch(128),   32,  "ch(128)"),
    (ch(256),   64,  "ch(256)"),
    (ch(512),   128, "ch(512)"),
    (ch(1024),  256, "ch(1024)"),
    (dep(2),    1,   "dep(2)"),
]
for got, expected, name in checks:
    status = "✅" if got == expected else "❌"
    print(f"  {status} yolo11n {name} = {got} (expected {expected})")



--- Step 3: get_model_params Check ---
  ✅ yolo11n ch(64) = 16 (expected 16)
  ✅ yolo11n ch(128) = 32 (expected 32)
  ✅ yolo11n ch(256) = 64 (expected 64)
  ✅ yolo11n ch(512) = 128 (expected 128)
  ✅ yolo11n ch(1024) = 256 (expected 256)
  ✅ yolo11n dep(2) = 1 (expected 1)


In [6]:
# ── Step 4: verify Backbone ───────────────────────────────────────────────────
print("\n--- Step 4: Backbone Check ---")
dummy    = torch.zeros(1, 3, 640, 640)
backbone = Backbone("yolo11n")
p3, p4, p5 = backbone(dummy)
backbone_checks = [
    (list(p3.shape), [1, 128,  80, 80], "P3"),
    (list(p4.shape), [1, 128,  40, 40], "P4"),
    (list(p5.shape), [1, 256,  20, 20], "P5"),
]
for got, expected, name in backbone_checks:
    status = "✅" if got == expected else "❌"
    print(f"  {status} {name} : {got} (expected {expected})")


--- Step 4: Backbone Check ---
  ✅ P3 : [1, 128, 80, 80] (expected [1, 128, 80, 80])
  ✅ P4 : [1, 128, 40, 40] (expected [1, 128, 40, 40])
  ✅ P5 : [1, 256, 20, 20] (expected [1, 256, 20, 20])


In [7]:
# ── Step 5: verify Neck ───────────────────────────────────────────────────────
print("\n--- Step 5: Neck Check ---")
neck     = Neck("yolo11n")
n3, n4, n5 = neck(p3, p4, p5)
neck_checks = [
    (list(n3.shape), [1,  64, 80, 80], "n3"),
    (list(n4.shape), [1, 128, 40, 40], "n4"),
    (list(n5.shape), [1, 256, 20, 20], "n5"),
]
for got, expected, name in neck_checks:
    status = "✅" if got == expected else "❌"
    print(f"  {status} {name} : {got} (expected {expected})")


--- Step 5: Neck Check ---
  ✅ n3 : [1, 64, 80, 80] (expected [1, 64, 80, 80])
  ✅ n4 : [1, 128, 40, 40] (expected [1, 128, 40, 40])
  ✅ n5 : [1, 256, 20, 20] (expected [1, 256, 20, 20])


In [8]:
# ── Step 6: verify DetectHead ─────────────────────────────────────────────────
print("\n--- Step 6: DetectHead Check ---")
detecthead = DetectHead("yolo11n", nc=1)
print(f"  ✅ Strides : {detecthead.detect.stride.tolist()}")

detecthead.eval()
with torch.no_grad():
    out = detecthead([n3, n4, n5])
status = "✅" if list(out.shape) == [1, 5, 8400] else "❌"
print(f"  {status} Inference : {list(out.shape)} (expected [1, 5, 8400])")

detecthead.train()
out_train = detecthead([n3, n4, n5])
checks = [
    (list(out_train[0].shape), [1, 65, 80, 80], "Scale 0"),
    (list(out_train[1].shape), [1, 65, 40, 40], "Scale 1"),
    (list(out_train[2].shape), [1, 65, 20, 20], "Scale 2"),
]
for got, expected, name in checks:
    status = "✅" if got == expected else "❌"
    print(f"  {status} {name} : {got} (expected {expected})")


--- Step 6: DetectHead Check ---
  ✅ Strides : [8.0, 16.0, 32.0]
  ✅ Inference : [1, 5, 8400] (expected [1, 5, 8400])
  ✅ Scale 0 : [1, 65, 80, 80] (expected [1, 65, 80, 80])
  ✅ Scale 1 : [1, 65, 40, 40] (expected [1, 65, 40, 40])
  ✅ Scale 2 : [1, 65, 20, 20] (expected [1, 65, 20, 20])


In [9]:
# ── Step 7: verify YOLOv11 all variants ──────────────────────────────────────
print("\n--- Step 7: YOLOv11 All Variants Check ---")
for name in ["yolo11n", "yolo11s", "yolo11m", "yolo11l", "yolo11x"]:
    try:
        m = YOLOv11(name, nc=1)
        m.eval()
        with torch.no_grad():
            o = m(dummy)
        status = "✅" if list(o.shape) == [1, 5, 8400] else "❌"
        params = sum(p.numel() for p in m.parameters())
        print(f"  {status} {name:<10} → {list(o.shape)}  params: {params:,}")
    except Exception as e:
        print(f"  ❌ {name:<10} → Error: {e}")


--- Step 7: YOLOv11 All Variants Check ---
  ✅ yolo11n    → [1, 5, 8400]  params: 2,317,523
  ✅ yolo11s    → [1, 5, 8400]  params: 8,340,435
  ✅ yolo11m    → [1, 5, 8400]  params: 17,973,971
  ✅ yolo11l    → [1, 5, 8400]  params: 21,676,947
  ✅ yolo11x    → [1, 5, 8400]  params: 25,293,427
